# Econ 5200 PS2 Warm-up — the code PS2 asks you to write

**Not graded.** Practice for Problem Set 2, on different data. Each exercise
shows you working code (*Watch*), asks you to write a similar piece yourself
(*Your turn*), then checks it. The exercises follow PS2's order: a prior you control, select on significance, split by power, bootstrap a selected total, then a rule passed as a function.

- **Turn AI code completion off** (Colab: *Settings › AI assistance*), as PS2
  Phases 1–3 require. You may still click **Explain error**.
- The *Watch* cells use drug trials and the *Your turn* cells use marketing campaigns. The two
  differ **on purpose**: you write each piece yourself instead of copying it.
- Run the cells in order: later exercises use what earlier ones made.
- If a check fails, read its message, fix your cell, and run both again.
- Stuck for more than five minutes? Look back at the *Watch* cell above it.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

---
## 1. A prior you control (PS2 Phase 1.1)

To audit a decision rule you need the truth for every unit. A mixture prior: most effects exactly zero, a minority real. Draw who is real with `rng.random(k) < share`, then `np.where(real, draw_if_real, 0.0)`. Give each unit its own sample size with `rng.integers(low, high, size=k)`. The SE of a difference in two means with outcome sd σ and n per arm is `σ * np.sqrt(2 / n)`, and the estimate is the truth plus `Normal(0, se)`.

**Watch:** run this and read it.

In [ ]:
rng = np.random.default_rng(3)
k = 150                                                           # 150 drug trials
real = rng.random(k) < 0.25
true_eff = np.where(real, rng.normal(0.30, 0.10, size=k), 0.0)    # in outcome-sd units
n = rng.integers(40, 400, size=k)                                 # patients per arm
se = 1.0 * np.sqrt(2 / n)
trials = pd.DataFrame({"true": true_eff, "n": n, "se": se, "est": true_eff + rng.normal(0, se)})

print(f"share exactly zero {(trials['true'] == 0).mean():.2f}")
print(trials.describe().loc[["mean", "min", "max"]].round(3))

**Your turn:** build `campaigns`: 400 marketing campaigns, 10% with a real lift drawn from Normal(0.08, 0.03) and the rest exactly 0, users per arm drawn from 500 up to 20,000, outcome sd 1.0. Columns `true`, `n`, `se`, `est`, as above.

In [ ]:
rng = np.random.default_rng(5)
# YOUR CODE HERE
campaigns = None

In [ ]:
assert campaigns is not None and len(campaigns) == 400, "campaigns needs 400 rows"
assert set(["true", "n", "se", "est"]) <= set(campaigns.columns), "columns: true, n, se, est"
assert 0.85 < (campaigns["true"] == 0).mean() < 0.95, "about 90% of lifts should be exactly 0"
assert np.allclose(campaigns["se"], np.sqrt(2 / campaigns["n"])), "se = 1.0 * sqrt(2 / n)"
z = (campaigns["est"] - campaigns["true"]) / campaigns["se"]
assert 0.85 < z.std() < 1.15, "est should be true + Normal(0, se): one draw per campaign, with that campaign's se"
print(f"{(campaigns['true'] > 0).sum()} real lifts out of 400; n from {campaigns['n'].min():,} to {campaigns['n'].max():,}")

---
## 2. Select on significance, then compare truth with estimate (PS2 Phases 1.2 and 2.1)

A two-sided p-value from an estimate and its SE is `2 * stats.norm.sf(abs(est / se))`. A True/False mask keeps the "winners". Among the winners, compare the average estimate with the average truth: their ratio is the exaggeration, and the share of winners whose truth is 0 is the false-ship rate.

**Watch:** run this and read it.

In [ ]:
trials["p"] = 2 * stats.norm.sf(np.abs(trials["est"] / trials["se"]))
won = trials["p"] < 0.05
w = trials[won]
print(f"{won.sum()} trials 'work' | true effect 0 in {(w['true'] == 0).mean():.0%} of them")
print(f"mean estimate {w['est'].mean():.3f} vs mean truth {w['true'].mean():.3f}: ratio {w['est'].mean() / w['true'].mean():.2f}")

**Your turn:** on `campaigns`, add the column `p`, keep the mask `shipped` for p < 0.05 **and a positive estimate** (nobody ships a loss), and compute `false_share` (share of shipped with true lift 0) and `exaggeration` (mean estimate over mean truth, among shipped).

In [ ]:
# YOUR CODE HERE
shipped = false_share = exaggeration = None

In [ ]:
assert "p" in campaigns.columns and campaigns["p"].between(0, 1).all(), "p is a column of p-values in [0, 1]"
assert shipped is not None and shipped.dtype == bool and len(shipped) == 400, "shipped is one True/False per campaign"
assert (campaigns.loc[shipped, "est"] > 0).all(), "only positive estimates ship"
assert 0 <= false_share <= 1, "false_share is a share"
assert exaggeration > 1, "selecting on significance should make the estimates run ABOVE the truth"
print(f"{shipped.sum()} ship | {false_share:.0%} of them do nothing | estimates {exaggeration:.2f}× the truth")

---
## 3. Split by power (PS2 Phase 2.1)

Each unit's power against a reference effect δ, for a one-sided 5% test, is close to `stats.norm.sf(1.645 - δ / se)`. `pd.qcut(x, 3, labels=[...])` cuts a column into thirds by value, and `groupby(...)` summarises each third. `observed=True` keeps only the tiers that exist.

**Watch:** run this and read it.

In [ ]:
trials["power"] = stats.norm.sf(1.645 - 0.30 / trials["se"])
trials["tier"] = pd.qcut(trials["power"], 3, labels=["low", "mid", "high"])
w = trials[won]
print(w.groupby("tier", observed=True).agg(shipped=("est", "size"), mean_est=("est", "mean"), mean_true=("true", "mean")).round(3))

**Your turn:** on `campaigns`, compute `power` against δ = 0.08, cut it into thirds as `tier` (low, mid, high), and build `by_tier`: for the shipped campaigns, one row per tier with the columns `n_shipped`, `mean_est`, `mean_true` and `ratio` (mean_est / mean_true).

In [ ]:
# YOUR CODE HERE
by_tier = None

In [ ]:
assert by_tier is not None and {"n_shipped", "mean_est", "mean_true", "ratio"} <= set(by_tier.columns), \
    "by_tier needs n_shipped, mean_est, mean_true and ratio"
assert campaigns["tier"].value_counts().between(130, 137).all(), "qcut into thirds: about 133 campaigns per tier"
assert by_tier["n_shipped"].sum() == shipped.sum(), "every shipped campaign belongs to one tier"
print(by_tier.round(3))
print("Which tier exaggerates most, and why does it need the most luck to ship?")

---
## 4. Bootstrap a selected total (PS2 Phase 2.2)

When the statistic is a total over a selected subset, resample the **units** and re-apply the selection inside every resample. By hand: draw row positions with `rng.integers(0, k, size=k)`. With scipy: bootstrap the row positions themselves, `stats.bootstrap((np.arange(k),), stat, vectorized=False, method="BCa")`, where `stat(idx)` recomputes the total on those rows.

**Watch:** run this and read it.

In [ ]:
def wins_total(df):
    """Sum of TRUE effects among the trials that would be declared to work."""
    return df.loc[df["p"] < 0.05, "true"].sum()

k = len(trials)
boot = np.array([wins_total(trials.iloc[rng.integers(0, k, size=k)]) for _ in range(2_000)])
res = stats.bootstrap((np.arange(k),), lambda idx: wins_total(trials.iloc[idx]), n_resamples=2_000,
                      vectorized=False, method="BCa", random_state=1)
print(f"total {wins_total(trials):.2f} | by hand [{np.percentile(boot, 2.5):.2f}, {np.percentile(boot, 97.5):.2f}]"
      f" | BCa [{res.confidence_interval.low:.2f}, {res.confidence_interval.high:.2f}]")

**Your turn:** write `shipped_total(df)` (the sum of true lift among campaigns with p < 0.05 and a positive estimate), bootstrap it by hand over the 400 campaigns (2,000 resamples) into `boot_totals` with its 95% percentile interval `lo, hi`, then get the BCa interval as `bca`.

In [ ]:
# YOUR CODE HERE
def shipped_total(df):
    pass

boot_totals = None
lo = hi = None
bca = None

In [ ]:
total = shipped_total(campaigns)
assert total is not None, "shipped_total should return a number: the sum of true lift it ships"
assert abs(total - campaigns.loc[shipped, "true"].sum()) < 1e-12, "shipped_total should match the shipped mask from exercise 2"
assert boot_totals is not None and len(boot_totals) == 2_000, "2,000 bootstrap totals"
assert lo < total < hi, "the percentile interval should contain the observed total"
assert bca is not None and bca.low < total < bca.high, "the BCa interval should contain it too"
print(f"realised true lift {total:.3f} | percentile [{lo:.3f}, {hi:.3f}] | BCa [{bca.low:.3f}, {bca.high:.3f}]")

---
## 5. A rule passed as a function, in a module (PS2 Phases 3.1 and 3.2)

A decision rule can be a function that takes the log and returns a True/False mask; then one `evaluate`-style function works for every rule. `%%writefile` saves a cell as a `.py` file, the block under `if __name__ == "__main__":` runs only when the file is run as a script, and `assert` lines state what must hold.

**Watch:** run this and read it.

In [ ]:
%%writefile trial_rules.py
"""Decision rules for a log of trials: each rule maps the log to a True/False mask."""
import pandas as pd


def p_rule(log: pd.DataFrame, alpha: float = 0.05) -> pd.Series:
    """Declare a trial a success when p < alpha."""
    return log["p"] < alpha


def count_true_zero(log: pd.DataFrame, rule) -> int:
    """How many trials the rule passes whose true effect is 0."""
    return int((log.loc[rule(log), "true"] == 0).sum())


if __name__ == "__main__":
    demo = pd.DataFrame({"p": [0.01, 0.20, 0.03], "true": [0.0, 0.3, 0.2]})
    assert p_rule(demo).tolist() == [True, False, True]
    assert count_true_zero(demo, p_rule) == 1
    print("trial_rules: demo passed")

**Your turn:** write `campaign_rules.py` with `n_floor(log, min_n=5000)` (True where p < 0.05, the estimate is positive **and** n ≥ min_n) and `realised(log, rule)` (the sum of true lift the rule ships), a `__main__` demo with at least two asserts, then the import cell below runs both rules.

In [ ]:
%%writefile campaign_rules.py
# YOUR CODE HERE

In [ ]:
import importlib
import campaign_rules
importlib.reload(campaign_rules)          # picks up your latest %%writefile
assert hasattr(campaign_rules, "n_floor") and hasattr(campaign_rules, "realised"), "campaign_rules needs n_floor and realised"
p_only = lambda log: (log["p"] < 0.05) & (log["est"] > 0)
assert campaign_rules.n_floor(campaigns).sum() <= p_only(campaigns).sum(), "a floor can only ship fewer campaigns"
assert abs(campaign_rules.realised(campaigns, p_only) - shipped_total(campaigns)) < 1e-12, "realised should match exercise 4"
!python campaign_rules.py
print(f"p < 0.05 alone realises {campaign_rules.realised(campaigns, p_only):.3f}; "
      f"with the n floor {campaign_rules.realised(campaigns, campaign_rules.n_floor):.3f}")

---
**Done.** A prior you control, selection on significance, tiers by power, a bootstrap of a selected total, and a rule passed as a function inside a tested module: the building blocks of PS2 Phases 1–3. Now open `ps2-starter.ipynb`.